# Figure 2 — Distinct perturbation programs converge on failed endothelial differentiation

Panels a and d–f are implemented. Panel b remains deferred pending resolution of the canonical EC-output column and sign convention. Panel c remains an editable schematic asset.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from src.cc_perturbseq.figure_notebook import get_path, initialize_figure, load_figure_manifest, panel_directory, source_asset_status, write_receipt
from src.cc_perturbseq.program_panels import *

def export(fig, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix, kwargs in [("pdf", {}), ("svg", {}), ("png", {"dpi": 300})]:
        path = out / f"{stem}.{suffix}"
        fig.savefig(path, **kwargs)
        result[suffix] = str(path)
    return result

FIGURE = load_figure_manifest("Figure_02")
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Panel a — pairwise target correlation


In [ ]:
effect_path = get_path(ctx["paths"], "programde.effect_matrix_all_days")
q_path = get_path(ctx["paths"], "programde.qvalue_matrix_all_days", required=False)
filename = config_value(ctx["paths"], "programde.effects_filename", required=False)
effects = load_program_effects(effect_path, q_path, filename=filename)
TARGET_GROUPS = {"BMP/TGFβ": ["BMPR1A","SMAD1","SMAD5","SMAD4"], "Mesoderm": ["MIXL1","EOMES"], "LM + EC": ["KDR","KCTD10","PRKCD","ETV2","PLCG1"], "Hippo": ["TAOK1","AMOTL2","LATS2","NF2"]}
fig, correlation, target_order = plot_target_effect_correlation(effects, target_groups=TARGET_GROUPS, min_features=10)
outputs = export(fig, "a", "Figure_02a_perturbation_effect_correlation_heatmap")
tables = save_source_tables({"target_correlation": correlation, "target_order": pd.DataFrame({"target": target_order})}, panel_directory(ctx, "a"), "Figure_02a")
write_receipt(ctx, "a", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"effects": str(effect_path)}, outputs={**outputs, **tables}, parameters={"distance": "correlation", "linkage": "average", "min_features": 10})
plt.show()


## Panel b — deferred input choice

The plotting implementation is retained in `program_figure2.py`, but execution is deferred until the canonical Day 3 EC-output field and its sign convention are fixed.


## Panel c — schematic


In [ ]:
source_asset_status(ctx, "c", "assets.figure2c_schematic")


## Panel d — guide-level EC fractions


In [ ]:
ec_path = get_path(ctx["paths"], "ccperturbseq.ec_fraction_table")
ec_table = standardize_ec_fraction_table(ec_path)
targets_d = ["ETV2", "KDR", "PLCG1", "PRKCD"]
fig, source_d = plot_guide_ec_fraction(ec_table, targets=targets_d)
outputs = export(fig, "d", "Figure_02d_guide_ec_fraction")
tables = save_source_tables({"guide_ec_fraction": source_d}, panel_directory(ctx, "d"), "Figure_02d")
write_receipt(ctx, "d", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"ec_fraction_table": str(ec_path)}, outputs={**outputs, **tables}, parameters={"targets": targets_d, "point_unit": "guide"})
plt.show()


## Panel e — P17/P27 and marker UMAPs


In [ ]:
adata_path = get_path(ctx["paths"], "ccperturbseq.integrated_anndata")
usage_path = get_path(ctx["paths"], "ccperturbseq.cnmf_usage")
adata, usage = load_integrated_anndata(adata_path, usage_path)
features_e = [("program",17),("program",27),("gene","SLC8A1"),("gene","GATA4")]
fig, source_e = plot_umap_feature_grid(adata, usage, features_e)
outputs = export(fig, "e", "Figure_02e_program_marker_umaps")
tables = save_source_tables(source_e, panel_directory(ctx, "e"), "Figure_02e")
write_receipt(ctx, "e", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata": str(adata_path), "usage": str(usage_path)}, outputs={**outputs, **tables}, parameters={"features": features_e})
plt.show()


## Panel f — carrier-seeded neighborhood enrichment


In [ ]:
guide_metadata_path = get_path(ctx["paths"], "ccperturbseq.target_metadata", required=False)
guide_map = load_guide_target_map(adata, guide_metadata_path)
targets_f = ["ETV2", "KDR", "PRKCD", "PLCG1"]
neighborhoods = carrier_seeded_neighborhood(adata, usage, guide_map, targets_f, k=300, pseudocount=0.5)
fig = plot_carrier_neighborhood_grid(adata, neighborhoods, targets_f)
outputs = export(fig, "f", "Figure_02f_carrier_seeded_neighborhoods")
tables = save_source_tables({"neighborhood_statistics": neighborhoods}, panel_directory(ctx, "f"), "Figure_02f")
write_receipt(ctx, "f", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata": str(adata_path), "usage": str(usage_path), "guide_metadata": str(guide_metadata_path)}, outputs={**outputs, **tables}, parameters={"targets": targets_f, "k": 300, "distance": "Euclidean on z-scored 50-program usage", "same_day": True, "pseudocount": 0.5})
plt.show()
